# API Best Practices — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/llm-engineering/4-3/api-best-practices.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/llm-engineering/4-3/api-best-practices.ipynb)

Companion to: [API Best Practices](https://anacodicai.org/learn/llm-engineering/4-3/api-best-practices)


## What you'll build

1. A **provider-aware truncation** function that respects UTF-8 character boundaries.
2. A demonstration that the SAME request survives intact on one provider tier and
   gets cut to under a third of its size on another.
3. A **retry-with-backoff-and-jitter** loop, and a worst-case wait-time calculation.

**No API key needed.** Everything below is pure Python standard library.

### Setup

In [1]:
import random
print("ready — standard library only")

ready — standard library only


## Step 1 — provider limits and a safe truncator

Different provider tiers accept different maximum request sizes. Truncating text
by raw byte count is unsafe: a UTF-8 character can span multiple bytes, so an
arbitrary cut can land in the middle of one. `truncate_utf8_bytes` backs up
byte-by-byte until the cut point decodes cleanly.

In [2]:
PROVIDER_LIMITS = {
    "fast-tier":    {"max_input_bytes": 3500,  "max_output_tokens": 1024},
    "premium-tier": {"max_input_bytes": 12000, "max_output_tokens": 4096},
}

def truncate_utf8_bytes(text, max_bytes):
    encoded = text.encode("utf-8")
    if len(encoded) <= max_bytes:
        return text, False
    cut = encoded[:max_bytes]
    while True:
        try:
            return cut.decode("utf-8"), True
        except UnicodeDecodeError:
            cut = cut[:-1]

def build_prompt(instructions, context_docs):
    parts = [instructions, ""]
    for i, doc in enumerate(context_docs):
        parts.append(f"[source {i}] {doc}")
    return "\n".join(parts)

def fit_to_provider(prompt, provider):
    limit = PROVIDER_LIMITS[provider]["max_input_bytes"]
    fitted, was_truncated = truncate_utf8_bytes(prompt, limit)
    return fitted, was_truncated, len(prompt.encode("utf-8")), limit

print("provider limits and truncation helpers ready")

provider limits and truncation helpers ready


## Step 2 — the same request, two tiers

**Try it:** change the `* 20` repeat factor below (fewer repeats = a shorter
prompt) and re-run — watch the `fast-tier` "kept %" change as the prompt grows
past and shrinks below its 3500-byte limit.

In [3]:
instructions = "Answer the operator's question using only the sources below."
context_docs = [
    "The staging cluster autoscaler targets 65% average CPU across nodes "
    "and adds one node per scale-up event, waiting 180 seconds between "
    "events to avoid thrashing on transient spikes.",
    "Node drains during a scale-down wait for in-flight requests to "
    "complete for up to 30 seconds before forcibly terminating remaining "
    "connections, and this is configurable per node pool.",
    "Scale-up events are suppressed for 10 minutes after any node pool "
    "resize failure, to avoid retrying against a quota limit that has "
    "not yet been lifted by the cloud provider.",
] * 20   # <-- change this repeat count and re-run

prompt = build_prompt(instructions, context_docs)
print(f"Full prompt size: {len(prompt.encode('utf-8'))} bytes\n")

for provider in PROVIDER_LIMITS:
    fitted, truncated, original_size, limit = fit_to_provider(prompt, provider)
    kept_pct = 100 * len(fitted.encode("utf-8")) / original_size
    print(f"{provider:12} limit={limit:6} bytes  "
          f"truncated={str(truncated):5}  kept={kept_pct:5.1f}% of original")

Full prompt size: 11551 bytes

fast-tier    limit=  3500 bytes  truncated=True   kept= 30.3% of original
premium-tier limit= 12000 bytes  truncated=False  kept=100.0% of original


## Step 3 — the UTF-8 boundary check

A cut that lands mid-character has to back up. Here we force exactly that
situation on purpose (every 'é' is 2 bytes) and confirm the result still
decodes cleanly.

In [4]:
tricky = "café " * 400
fitted, was_truncated = truncate_utf8_bytes(tricky, 101)
assert was_truncated
assert fitted.encode("utf-8")  # raises if the cut left an invalid boundary
print(f"UTF-8 boundary check: kept {len(fitted.encode('utf-8'))} of "
      f"{len(tricky.encode('utf-8'))} bytes, decodes cleanly: True")

UTF-8 boundary check: kept 101 of 2400 bytes, decodes cleanly: True


## Step 4 — retrying a rate-limited call with backoff + jitter

```
delay(attempt) = base_delay * 2^attempt + jitter
```

**Try it:** change `max_retries` or `base_delay` and re-run the `fail_first_n=5`
case — watch it start succeeding once retries exceed 5, or keep failing if you
lower `max_retries` below that.

In [5]:
class RateLimited(Exception):
    pass

def call_with_backoff(fn, max_retries=4, base_delay=0.5, rng=None):
    rng = rng or random.Random(7)
    attempt = 0
    delays = []
    while True:
        try:
            return fn(), attempt, delays
        except RateLimited:
            if attempt >= max_retries:
                raise
            delay = base_delay * (2 ** attempt) + rng.uniform(0, 0.1)
            delays.append(round(delay, 3))
            attempt += 1

def flaky_call(fail_first_n):
    count = {"n": 0}
    def _call():
        count["n"] += 1
        if count["n"] <= fail_first_n:
            raise RateLimited()
        return f"ok after {count['n']} attempt(s)"
    return _call

for fail_first_n in (0, 2, 5):
    try:
        call = flaky_call(fail_first_n)
        result, attempt, delays = call_with_backoff(call, max_retries=4, base_delay=0.5)
        print(f"fail_first_n={fail_first_n}: {result}, retries used={attempt}, "
              f"backoff delays={delays}")
    except RateLimited:
        print(f"fail_first_n={fail_first_n}: gave up after retries exhausted")

fail_first_n=0: ok after 1 attempt(s), retries used=0, backoff delays=[]
fail_first_n=2: ok after 3 attempt(s), retries used=2, backoff delays=[0.532, 1.015]
fail_first_n=5: gave up after retries exhausted


## Step 5 — the worst-case wait, as a number you'd actually budget for

```
total = base_delay * (2^0 + 2^1 + ... + 2^(max_retries-1))
```

In [6]:
base_delay = 0.5
max_retries = 4
worst_case_total = base_delay * sum(2 ** i for i in range(max_retries))
print(f"With base_delay={base_delay}s and max_retries={max_retries}, "
      f"worst-case total wait before giving up: {worst_case_total:.1f} seconds")
print("(ignoring the small jitter term added on top of each delay)")

With base_delay=0.5s and max_retries=4, worst-case total wait before giving up: 7.5 seconds
(ignoring the small jitter term added on top of each delay)


## Try it yourself

- Push `context_docs`'s repeat factor down to `* 3` in Step 2 — at what repeat
  count does `fast-tier` stop truncating at all?
- In Step 4, set `max_retries=2` and re-run the `fail_first_n=5` case — confirm
  it now raises `RateLimited` instead of succeeding.
- In Step 5, double `base_delay` and see how fast the worst-case wait grows —
  it's exponential, not linear.